In [ ]:
!pip install transformers faiss-cpu datasets

## FIR Dataset

In [ ]:
import pandas as pd

df = pd.read_excel(
    "Synthetic_data.xlsx"
)

print(df.head())

                                                Text  \
0  بخدمت جناب ایس ایچ او صاحب۔ مسمی [مدعی] نے حاض...   
1  جناب عالی! گزارش ہے کہ [مدعی] اپنے کاروباری پت...   
2  رپورٹ بحضور افسر محترم! [مدعی] نے تحریری درخوا...   
3  تفصیلِ مقدمہ یہ ہے کہ خفیہ اطلاع پر سرکار کی ج...   
4  باعرضِ احوال، مسمی [مدعی] نے رپورٹ درج کروائی ...   

                                       Trigger_words Applied_Sections  \
0       چوری، گھر میں داخلہ، تالے توڑ کر، مال مسروقہ         379/ 380   
1  اسلحہ سے لیس، زبردستی چھین، جان سے مارنے کی دھ...    382/ 392/ 506   
2  جعلی دستاویز، بددیانتی، دھوکہ دہی، مالیاتی دھو...              420   
3                 منشیات، چرس، برآمد، چھاپہ مارا گیا         9C CNSA    
4           نابالغ لڑکی، اغوا، زبردستی، جنسی استحصال         364/ 366   

      Crime_category  
0               چوری  
1              ڈکیتی  
2  مالیاتی دھوکہ دہی  
3             منشیات  
4               اغوا  


In [ ]:
print(df.columns)

Index(['Text', 'Trigger_words', 'Applied_Sections', 'Crime_category'], dtype='object')


## Embeddings

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import faiss
from datasets import Dataset

# Initialize a tokenizer and model to generate embeddings
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
df["Text"]

,Text
0,بخدمت جناب ایس ایچ او صاحب۔ مسمی [مدعی] نے حاض...
1,جناب عالی! گزارش ہے کہ [مدعی] اپنے کاروباری پت...
2,رپورٹ بحضور افسر محترم! [مدعی] نے تحریری درخوا...
3,تفصیلِ مقدمہ یہ ہے کہ خفیہ اطلاع پر سرکار کی ج...
4,باعرضِ احوال، مسمی [مدعی] نے رپورٹ درج کروائی ...
...,...
3162,زبانی [مدعی] بمطابق خانہ نمبر 02 پستہ بالا پر ...
3163,زبانی [مدعیہ] بمطابق خانہ نمبر 02 [مدعی پتہ] پ...
3164,من [اہلکار] ہمراہی ملازمان سرکاری موبائل پر عل...
3165,بمطابق رپورٹ مورخہ 14/03/2025، مدعی جو کہ [پرا...


**Embedding Generation -with all headers**

In [ ]:
import torch
import numpy as np
import gc

# =========================
# DEVICE SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# =========================
# EMBEDDING FUNCTION (BATCHED)
# =========================
def embed_text_batch(texts, batch_size=16):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        # move to device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)

        all_embeddings.append(embeddings.cpu().numpy())

        # memory cleanup
        del inputs, outputs, embeddings
        torch.cuda.empty_cache()
        gc.collect()

    return np.vstack(all_embeddings)

# =========================
# DATA PREP
# =========================
texts = (

    df["Text"].astype(str)

    + " "

    + df["Trigger_words"].astype(str)

    + " "

    + df["Crime_category"].astype(str)

    + " "

    + df["Applied_Sections"].astype(str)

).tolist()


# =========================
# RUN EMBEDDINGS
# =========================
embeddings = embed_text_batch(texts, batch_size=16)

# =========================
# OUTPUT CHECK
# =========================
print("Embeddings shape:", embeddings.shape)
print("First embedding:", embeddings[0])

Embeddings shape: (3167, 384)
First embedding: [-1.75046036e-03  2.34272912e-01 -3.75337811e-04 -7.91307688e-02
 -9.90776345e-03  2.53114123e-02  8.06229562e-03 -2.37270907e-01
  4.01972532e-02 -2.10590661e-01  7.71068692e-01 -1.56131163e-02
  3.11887920e-01  4.36978072e-01  3.02002609e-01  4.63021621e-02
  1.70080718e-02  1.76077455e-01 -3.24638747e-02  1.80730075e-01
  1.87281966e-02 -2.62217015e-01  1.83273315e-01 -4.47050124e-01
 -3.08333486e-02 -1.43741831e-01 -1.26892373e-01  7.60893077e-02
 -1.78901717e-01  1.01004019e-01  4.08079028e-02 -1.03622198e-01
 -2.81695753e-01 -3.88967782e-01 -3.73867720e-01  1.08195506e-01
  2.15469301e-01  3.31470549e-01  2.76349157e-01  1.10742450e-01
  2.03569129e-01 -7.51833394e-02  1.04941718e-01 -3.46755028e-01
 -9.25547704e-02  4.65970226e-02 -1.52851000e-01 -3.90058127e-03
 -2.17618361e-01 -3.26039523e-01 -1.35094076e-01 -1.29540358e-02
  3.04450482e-01  1.55552447e-01  2.59222716e-01 -3.13512772e-01
 -3.49607766e-01  7.23406523e-02  2.3317477

## indexing by FAISS

In [ ]:
# Initialize a FAISS index and add embeddings
#index = faiss.IndexFlatL2(embeddings.shape[1])  # L2 distance for similarity
#index.add(embeddings)  # Add document embeddings to the index

# ============================================
# NORMALIZE EMBEDDINGS
# ============================================

faiss.normalize_L2(embeddings)

# ============================================
# COSINE SIMILARITY INDEX
# ============================================

index = faiss.IndexFlatIP(

    embeddings.shape[1]
)

index.add(embeddings)


print("FAISS index created and embeddings added!")


FAISS index created and embeddings added!


In [ ]:
# ============================================
# SAVE EMBEDDINGS
# ============================================

import numpy as np

np.save(
    "sentence_transformal/embeddings.npy",
    embeddings
)

print("Embeddings Saved ")

# ============================================
# SAVE FAISS INDEX
# ============================================

faiss.write_index(
    index,
    "sentence_transformal/faiss_index.bin"
)

print("FAISS Index Saved ")

# ============================================
# DONE
# ============================================

print("\nSaved Files:")
print("embeddings.npy")
print("faiss_index.bin")

## Retrieval

**Full Legal Context Retrieval**

In [ ]:
# ============================================
# EXAMPLE QUERY
# ============================================

query = """
بینک سے رقم نکال کر گھر جاتے ہوئے
اسلحہ کے زور پر نقدی اور موٹر سائیکل چھین لی گئی۔

Trigger Words:
موٹر سائیکل، اسلحہ، چھیننا، نقدی، ڈکیتی
"""

# ============================================
# QUERY EMBEDDING
# ============================================

query_embedding = embed_text_batch(

    [query],

    batch_size=1
)

query_embedding = np.array(

    query_embedding,

    dtype="float32"
)
# ============================================
# NORMALIZE QUERY
# ============================================

faiss.normalize_L2(query_embedding)

# ============================================
# FAISS SEARCH
# ============================================

D, I = index.search(

    query_embedding,

    k=5
)

# ============================================
# RETRIEVE DOCUMENTS
# ============================================

retrieved_docs = [

    {

        "text":
        df.iloc[doc_idx]["Text"],

        "trigger_words":
        df.iloc[doc_idx]["Trigger_words"],

         "sections":
        df.iloc[doc_idx]["Applied_Sections"],


        "category":
        df.iloc[doc_idx]["Crime_category"],

        "similarity":
        D[0][sim_idx]
    }

    for sim_idx, doc_idx in enumerate(I[0])
]

# ============================================
# DISPLAY RESULTS
# ============================================

print("\nTOP 5 SIMILAR FIRs\n")

for i, doc in enumerate(retrieved_docs, 1):

    print("="*70)

    print(f"RESULT {i}")

    print("\nFIR:")
    print(doc["text"])

    print("\nTrigger_words:")
    print(doc["trigger_words"])

    print("\nApplied Sections:")
    print(doc["sections"])

    print("\nCrime Category:")
    print(doc["category"])

    print("\nSimilarity Score:")
    print(round(doc["similarity"],4))

    print("="*70)


TOP 5 SIMILAR FIRs

RESULT 1

FIR:
مسمی [ملزم] نے جعلی چیک اور بوگس دستخط کر کے مدعی کے بینک اکاؤنٹ سے رقم نکلوائی اور مالیاتی دھوکہ دہی کی ہے۔

Trigger_words:
جعلی چیک، بوگس دستخط، مالیاتی دھوکہ دہی

Applied Sections:
420/ 468/ 489F

Crime Category:
مالیاتی دھوکہ دہی

Similarity Score:
0.6807
RESULT 2

FIR:
پولیس نے چھاپہ مار کر مسمی [ملزم] کو چوری شدہ بائیکس کے کٹے ہوئے پرزے (مسروقہ مال) گودام میں گاہکوں کو بیچتے گرفتار کیا۔

Trigger_words:
چوری شدہ بائیکس، مسروقہ مال، گودام

Applied Sections:
411/ 413

Crime Category:
مسروقہ مال

Similarity Score:
0.6555
RESULT 3

FIR:
مسمی [ملزم] کو چوری شدہ ائیر کنڈیشنر اور تانبے کے پائپ (مسروقہ مال) سستے داموں مارکیٹ میں فروخت کرتے ہوئے گرفتار کیا گیا۔

Trigger_words:
چوری شدہ اے سی، مسروقہ مال، فروخت کرتے ہوئے

Applied Sections:
411

Crime Category:
مسروقہ مال

Similarity Score:
0.654
RESULT 4

FIR:
مسمی [ملزم] کو چوری شدہ ائیر کنڈیشنر اور تانبے کے پائپ (مسروقہ مال) سستے داموں مارکیٹ میں فروخت کرتے ہوئے گرفتار کیا گیا۔

Trigger_words:
چوری شدہ 

#**Generation**

In [ ]:
!pip install google-generativeai


In [ ]:
import google.generativeai as genai

from google.colab import userdata

#GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

#genai.configure(api_key=GOOGLE_API_KEY)

genai.configure(api_key="Ac")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
for m in genai.list_models():
  # Print models that support the generateContent method
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [ ]:
#llm_model = "models/gemini-2.5-flash"
llm_model = "models/gemma-4-31b-it"

genai_model = genai.GenerativeModel(llm_model)


In [ ]:
#context_docs = [doc["text"] for doc in retrieved_docs]
# ============================================
# BUILD CONTEXT FROM RETRIEVED FIRs
# ============================================

context = ""

for i, doc in enumerate(retrieved_docs, 1):

    context += f"""

Similar FIR {i}

FIR:
{doc['text']}

Trigger Words:
{doc['trigger_words']}

Applied Sections:
{doc['sections']}

Crime Category:
{doc['category']}

Similarity Score:
{round(float(doc['similarity']),4)}

----------------------------------------
"""



In [ ]:
#prompt = f"Query: {query}\nContext: {retrieved_docs}\nAnswer:"
#print(prompt)

# ============================================
# FINAL PROMPT
# ============================================

prompt = f"""

You are an expert Pakistani legal assistant.

Analyze the query FIR carefully.

Use the retrieved similar FIRs
to identify the most relevant PPC sections.

==================================================

QUERY FIR:

{query}

==================================================

SIMILAR FIRs:

{context}

==================================================

Return ONLY in this format:

Relevant Sections:
<sections>

Trigger Words:
<trigger words>

Legal Reasoning:
<reasoning>

Crime Category:
<category>

"""

# ============================================
# PRINT PROMPT
# ============================================

print(prompt)





You are an expert Pakistani legal assistant.

Analyze the query FIR carefully.

Use the retrieved similar FIRs
to identify the most relevant PPC sections.


QUERY FIR:


بینک سے رقم نکال کر گھر جاتے ہوئے
اسلحہ کے زور پر نقدی اور موٹر سائیکل چھین لی گئی۔

Trigger Words:
موٹر سائیکل، اسلحہ، چھیننا، نقدی، ڈکیتی



SIMILAR FIRs:



Similar FIR 1

FIR:
مسمی [ملزم] نے جعلی چیک اور بوگس دستخط کر کے مدعی کے بینک اکاؤنٹ سے رقم نکلوائی اور مالیاتی دھوکہ دہی کی ہے۔

Trigger Words:
جعلی چیک، بوگس دستخط، مالیاتی دھوکہ دہی

Applied Sections:
420/ 468/ 489F

Crime Category:
مالیاتی دھوکہ دہی

Similarity Score:
0.6807

----------------------------------------


Similar FIR 2

FIR:
پولیس نے چھاپہ مار کر مسمی [ملزم] کو چوری شدہ بائیکس کے کٹے ہوئے پرزے (مسروقہ مال) گودام میں گاہکوں کو بیچتے گرفتار کیا۔

Trigger Words:
چوری شدہ بائیکس، مسروقہ مال، گودام

Applied Sections:
411/ 413

Crime Category:
مسروقہ مال

Similarity Score:
0.6555

----------------------------------------


Similar FIR 3

FIR:
مسمی [

In [ ]:
#response = genai_model.generate_content(prompt)
#print(response.text)
# ============================================
# GENERATE RESPONSE
# ============================================

response = genai_model.generate_content(prompt)

# ============================================
# OUTPUT
# ============================================

print("\nLLM RESPONSE\n")

print(response.text)


LLM RESPONSE

*   Query FIR: "بینک سے رقم نکال کر گھر جاتے ہوئے اسلحہ کے زور پر نقدی اور موٹر سائیکل چھین لی گئی۔" (Translation: While going home after withdrawing money from the bank, cash and a motorcycle were snatched at gunpoint/force of arms.)
    *   Trigger Words provided in query: موٹر سائیکل (Motorcycle), اسلحہ (Arms/Weapon), چھیننا (Snatching), نقدی (Cash), ڈکیتی (Robbery).
    *   Core Action: Forcible taking of property (money and bike) using a weapon.

    *   Similar FIR 1: Financial fraud (420/468/489F) - Not relevant.
    *   Similar FIR 2: Possession of stolen property (411/413) - Related to aftermath, but not the act of snatching.
    *   Similar FIR 3, 4, 5: Possession of stolen property (411) - Not relevant to the act of robbery/snatching.
    *   *Conclusion:* None of the provided similar FIRs match the specific act of "armed robbery/snatching" described in the query. I must rely on expert knowledge of the Pakistan Penal Code (PPC).

    *   The act involves:
    

In [ ]:
#prompt = f"Query: {query}\nContext: {retrieved_docs}\nAnswer in a sentence:"
#response = genai_model.generate_content(prompt)
#print(response.text)
# ============================================
# LEGAL RAG PROMPT
# ============================================

prompt = f"""

You are an expert Pakistani legal assistant.

Analyze the FIR complaint carefully
using the retrieved similar FIRs.

==================================================

QUERY FIR:

{query}

==================================================

RETRIEVED SIMILAR FIRs:

{retrieved_docs}

==================================================

Answer ONLY in this format:

Relevant Sections:
<sections>

Trigger Words:
<trigger words>

Crime Category:
<category>

Legal Reasoning:
<reasoning>

"""

# ============================================
# GENERATE RESPONSE
# ============================================

response = genai_model.generate_content(prompt)

# ============================================
# OUTPUT
# ============================================

print("\nLLM RESPONSE\n")

print(response.text)





LLM RESPONSE

*   Query FIR: "بینک سے رقم نکال کر گھر جاتے ہوئے اسلحہ کے زور پر نقدی اور موٹر سائیکل چھین لی گئی۔" (Translation: While going home after withdrawing money from the bank, cash and a motorcycle were snatched at gunpoint/force of weapons.)
    *   Trigger Words provided in Query: موٹر سائیکل (Motorcycle), اسلحہ (Weapon), چھیننا (Snatching), نقدی (Cash), ڈکیتی (Robbery).
    *   Retrieved FIRs:
        1.  Financial fraud (fake checks) - Sections 420/468/489F. (Irrelevant)
        2.  Possession of stolen bike parts - Sections 411/413. (Relates to stolen property, but not the act of robbery).
        3.  Possession of stolen AC/pipes - Section 411. (Irrelevant)

    *   The act involves:
        1.  Use of a weapon (اسلحہ).
        2.  Force/Threat (زور پر).
        3.  Theft/Snatching (چھین لی).
        4.  Items stolen: Cash and Motorcycle.
    *   In the Pakistan Penal Code (PPC):
        *   Section 392: Punishment for robbery (Theft/Extortion becomes robbery if the off

In [ ]:

#using a different prompt (an example of QA)
#response = genai_model.generate_content(prompt)
#print(response.text)
# ============================================
# ANOTHER TEST QUERY
# ============================================

query2 = """
دفتر جاتے ہوئے شہری سے اسلحہ کے زور پر
موٹر سائیکل، موبائل اور نقدی چھین لی گئی۔
"""

prompt2 = f"""

You are an expert Pakistani legal assistant.

Analyze the FIR complaint carefully
using the retrieved similar FIRs.

==================================================

QUERY FIR:

{query2}

==================================================

RETRIEVED SIMILAR FIRs:

{retrieved_docs}

==================================================

Answer ONLY in this format:

Relevant Sections:
<sections>

Trigger Words:
<trigger words>

Crime Category:
<category>

Legal Reasoning:
<reasoning>

"""

response2 = genai_model.generate_content(prompt2)

print("\nSECOND QUERY RESPONSE\n")

print(response2.text)



SECOND QUERY RESPONSE

*   Role: Expert Pakistani legal assistant.
    *   Task: Analyze a specific FIR complaint and identify relevant legal sections, trigger words, crime category, and legal reasoning.
    *   Query FIR: "دفتر جاتے ہوئے شہری سے اسلحہ کے زور پر موٹر سائیکل، موبائل اور نقدی چھین لی گئی۔" (Translation: A citizen was robbed of a motorcycle, mobile phone, and cash at gunpoint while going to the office.)
    *   Retrieved FIRs: A list of cases involving financial fraud (420, 468, 489F) and possession of stolen property (411, 413).

    *   Act: Snatching/Robbery (چھین لی گئی).
    *   Means: Use of weapon (اسلحہ کے زور پر).
    *   Items stolen: Motorcycle, mobile, cash.
    *   Context: Public place/on the way to office.

    *   The retrieved FIRs are *not* directly relevant. One is about fake checks (420), and the others are about selling stolen property (411/413).
    *   The query FIR describes a *robbery/snatching* incident, not just the possession of stolen goods o

Testing


In [ ]:
import faiss
import numpy as np

# ============================================
# LOAD EMBEDDINGS
# ============================================

embeddings = np.load(
    "sentence_transformal/embeddings.npy"
)

print("Embeddings Loaded ")

# ============================================
# LOAD INDEX
# ============================================

index = faiss.read_index(
    "sentence_transformal/faiss_index.bin"
)

print("FAISS Index Loaded ")

print("Embeddings Shape:", embeddings.shape)
print("Total Vectors:", index.ntotal)

In [ ]:
# =========================================================
# MINI-LM + FAISS RAG EVALUATION (75 FIRs)
# =========================================================

!pip install -q transformers faiss-cpu openpyxl

import pandas as pd
import numpy as np
import faiss
import torch
import re

from transformers import AutoTokenizer, AutoModel

# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# =========================================================
# LOAD TEST FILE
# =========================================================

test_df = pd.read_csv(
    "200split/FIR_Test.csv"
)

print("Total Test FIRs:", len(test_df))

# =========================================================
# LOAD DATABASE
# =========================================================

df = pd.read_excel(
    "Synthetic_data.xlsx"
)

print("Total Database FIRs:", len(df))

# =========================================================
# LOAD MINILM
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

model = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
).to(device)

model.eval()

print("MiniLM Loaded ")


# ============================================
# LOAD EMBEDDINGS
# ============================================

embeddings = np.load(
    "sentence_transformal/embeddings.npy"
)

print("Embeddings Loaded ")

# ============================================
# LOAD INDEX
# ============================================

index = faiss.read_index(
    "sentence_transformal/faiss_index.bin"
)

print("FAISS Index Loaded ")

print("Embeddings Shape:", embeddings.shape)
print("Total Vectors:", index.ntotal)

# =========================================================
# QUERY EMBEDDING FUNCTION
# =========================================================

def embed_query(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        embedding = outputs.last_hidden_state.mean(
            dim=1
        )

    embedding = embedding.cpu().numpy()

    return embedding.astype("float32")

# =========================================================
# SECTION EXTRACTOR
# =========================================================

def extract_sections(text):

    sections = re.findall(
        r'\d+[A-Z\-]*',
        str(text)
    )

    return sorted(list(set(sections)))

# =========================================================
# RESULTS
# =========================================================

results = []

# =========================================================
# TEST LOOP
# =========================================================

for idx, row in test_df.iterrows():

    query = str(row["Text"])

    actual_sections = str(
        row["Applied_Sections"]
    )

    # =====================================================
    # QUERY EMBEDDING
    # =====================================================

    query_embedding = embed_query(query)

    faiss.normalize_L2(query_embedding)

    # =====================================================
    # SEARCH
    # =====================================================

    D, I = index.search(
        query_embedding,
        k=3
    )

    # =====================================================
    # RETRIEVED SECTIONS
    # =====================================================

    retrieved_sections = []

    for doc_idx in I[0]:

        retrieved_sections.append(

            str(
                df.iloc[doc_idx]["Applied_Sections"]
            )
        )

    combined_sections = " / ".join(
        retrieved_sections
    )

    # =====================================================
    # PREDICTION
    # =====================================================

    predicted_sections = extract_sections(
        combined_sections
    )

    actual_clean = extract_sections(
        actual_sections
    )

    exact_match = int(

        set(predicted_sections)

        ==

        set(actual_clean)
    )

    # =====================================================
    # SAVE
    # =====================================================

    results.append({

        "Query_FIR":
        query,

        "Actual_Sections":
        actual_sections,

        "Predicted_Sections":
        ", ".join(predicted_sections),

        "Retrieved_Sections":
        combined_sections,

        "Similarity_1":
        round(float(D[0][0]), 4),

        "Similarity_2":
        round(float(D[0][1]), 4),

        "Similarity_3":
        round(float(D[0][2]), 4),

        "Exact_Match":
        exact_match
    })

    print(
        f"Processed {idx+1}/{len(test_df)}"
    )

# =========================================================
# RESULTS
# =========================================================

results_df = pd.DataFrame(results)

accuracy = results_df[
    "Exact_Match"
].mean()

print("\n===================================")
print("RAG EVALUATION COMPLETE ")
print("===================================")

print(
    f"Exact Match Accuracy: {accuracy:.4f}"
)

# =========================================================
# SAVE
# =========================================================

save_path = (
    ""
    "minilm_rag_test_results.xlsx"
)

results_df.to_excel(
    save_path,
    index=False
)

print("\nResults Saved")
print(save_path)

In [ ]:
import pandas as pd

df = pd.read_excel("minilm_rag_test_results.xlsx")

df_75 = df.head(75)

total = len(df_75)
correct = df_75["Exact_Match"].sum()
accuracy = (correct / total) * 100

print("="*50)
print(f"Total Test Cases : {total}")
print(f"Correct Matches  : {correct}")
print(f"Wrong Matches    : {total - correct}")
print(f"Accuracy         : {accuracy:.2f}%")
print("="*50)

wrong_cases = df_75[df_75["Exact_Match"] == 0]

print("\nWRONG PREDICTIONS:\n")

for i, row in wrong_cases.iterrows():
    print("="*80)
    print(f"Query: {row['Query_FIR'][:300]}...")
    print(f"Actual Section     : {row['Actual_Sections']}")
    print(f"Predicted Section  : {row['Predicted_Sections']}")
    print(f"Retrieved Sections : {row['Retrieved_Sections']}")
    print(f"Similarity Scores  : {row['Similarity_1']}, "
          f"{row['Similarity_2']}, {row['Similarity_3']}")
    print("="*80)


Total Test Cases : 75
Correct Matches  : 2
Wrong Matches    : 73
Accuracy         : 2.67%

WRONG PREDICTIONS:

Query: بمطابق بیان زیر دفعہ 154 ض ف [مدعی] ساکن [مدعی پتہ] نے بعد دریافت پر بتلایا کہ پتہ بالا پر رہائش پذیر ہوں اور [پرائیویٹ ملازمت] کرتا ہوں میں نے مورخہ 24/12/2024 کو اپنی [موٹر سائیکل] نمبری [گاڑی نمبر] میکر [گاڑی کی کمپنی] برنگ سیاہ ماڈل 2016 انجن نمبر [کارڈ نمبر] چیس نمبر [کارڈ نمبر] کو بوقت ایک بجک...
Actual Section     : 381A
Predicted Section  : 381, 381A
Retrieved Sections : 381A / 381 / 381A
Similarity Scores  : 0.9855, 0.9827, 0.9825
Query: من [اہلکار] ہمراہی ملازمان سرکاری موبائل پر علاقہ گشت و انسدادِ جرائم میں مصروف تھا کہ دورانِ گشت بمقام [پتہ/مقام] پہنچا تو دو جوان العمر اشخاص کو مشکوک حالت میں کھڑے پایا۔ پولیس موبائل کو دیکھ کر انہوں نے فرار ہونے کی کوشش کی جنہیں ہمراہی ملازمان کی مدد سے بوقت 16:50 بجے پکڑ لیا گیا۔ دریافت پر انہو...
Actual Section     : 9A/9B CNSA
Predicted Section  : 23, 9B, 9C
Retrieved Sections : 9C CNSA  / 9B CNSA / SAA 23 (i) A
Similari

In [ ]:
import pandas as pd
import re

# Load Excel
df = pd.read_excel("minilm_top3_retrieval_results.xlsx")

def extract_sections(text):
    if pd.isna(text) or str(text).strip() == "0":
        return set()

    text = str(text).lower().replace(" ", "")

    tokens = re.findall(r'\d+[a-z]?|cnsa|saa|tip', text)

    final_set = set()

    for t in tokens:
        final_set.add(t)

        base = re.match(r'^(\d+)', t)
        if base:
            final_set.add(base.group(1))

    return final_set


def jaccard(actual, predicted):

    if pd.isna(predicted) or str(predicted).strip() == "0":
        return 0

    A = extract_sections(actual)
    B = extract_sections(predicted)

    union = A | B

    if len(union) == 0:
        return 0

    return round(len(A & B) / len(union), 4)


for i in [1,2,3]:

    col = f"Retrieved_Section_{i}"

    if col in df.columns:

        df[f"Jaccard_{i}"] = df.apply(
            lambda row: jaccard(
                row["Actual_Section"],
                row[col]
            ),
            axis=1
        )

    else:

        df[f"Jaccard_{i}"] = 0


df["Best_Jaccard"] = df[
    ["Jaccard_1","Jaccard_2","Jaccard_3"]
].max(axis=1)

print("Average Best Jaccard:", round(df["Best_Jaccard"].mean(),4))

df.to_excel(
    "minilm_rag_jaccard_results.xlsx",
    index=False
)

Average Best Jaccard: 0.4693


In [ ]:
import pandas as pd
import re

# Load file
df = pd.read_excel("minilm_top3_retrieval_results.xlsx")

# -----------------------------
# Extract Sections
# -----------------------------
def extract_sections(text):
    if pd.isna(text) or str(text).strip() == "0":
        return set()

    text = str(text).lower().replace(" ", "")

    tokens = re.findall(r'\d+[a-z]?|cnsa|saa|tip', text)

    final_set = set()

    for t in tokens:
        final_set.add(t)

        # Add base section (381a -> 381)
        m = re.match(r'^(\d+)', t)
        if m:
            final_set.add(m.group(1))

    return final_set

# -----------------------------
# Create universe of all labels
# -----------------------------
all_labels = set()

for col in ["Actual_Section",
            "Retrieved_Section_1",
            "Retrieved_Section_2",
            "Retrieved_Section_3"]:

    if col in df.columns:
        df[col].apply(lambda x: all_labels.update(extract_sections(x)))

all_labels = sorted(all_labels)

# -----------------------------
# Hamming Loss Function
# -----------------------------
def hamming_loss(actual, predicted):

    A = extract_sections(actual)
    B = extract_sections(predicted)

    if len(all_labels) == 0:
        return 0

    mismatches = 0

    for label in all_labels:
        actual_bit = label in A
        pred_bit = label in B

        if actual_bit != pred_bit:
            mismatches += 1

    return round(mismatches / len(all_labels), 4)

# -----------------------------
# Calculate Hamming Loss
# -----------------------------
for i in [1, 2, 3]:

    col = f"Retrieved_Section_{i}"

    if col in df.columns:
        df[f"Hamming_{i}"] = df.apply(
            lambda row: hamming_loss(
                row["Actual_Section"],
                row[col]
            ),
            axis=1
        )
    else:
        df[f"Hamming_{i}"] = 1

# -----------------------------
# Best Hamming Loss
# (Lower is Better)
# -----------------------------
df["Best_Hamming"] = df[
    ["Hamming_1", "Hamming_2", "Hamming_3"]
].min(axis=1)

print("Average Best Hamming Loss:",
      round(df["Best_Hamming"].mean(), 4))

# -----------------------------
# Save Result
# -----------------------------
df.to_excel(
    "minilm_rag_hamming_results.xlsx",
    index=False
)

Average Best Hamming Loss: 0.0313


In [ ]:
import pandas as pd
import re

# Load Excel
df = pd.read_excel("minilm_top3_retrieval_results.xlsx")


def extract_sections(text):
    if pd.isna(text) or str(text).strip() == "0":
        return set()

    text = str(text).lower().replace(" ", "")

    tokens = re.findall(r'\d+[a-z]?|cnsa|saa|tip', text)

    final_set = set()

    for t in tokens:
        final_set.add(t)

        # Add base section (381A -> 381)
        base = re.match(r'^(\d+)', t)
        if base:
            final_set.add(base.group(1))

    return final_set


def is_relevant(actual, retrieved):

    if pd.isna(retrieved) or str(retrieved).strip() == "0":
        return 0

    A = extract_sections(actual)
    B = extract_sections(retrieved)

    # Relevant if at least one PPC section matches
    return 1 if len(A & B) > 0 else 0


# Hit calculation
for i in [1, 2, 3]:

    col = f"Retrieved_Section_{i}"

    if col in df.columns:

        df[f"Hit_{i}"] = df.apply(
            lambda row: is_relevant(
                row["Actual_Section"],
                row[col]
            ),
            axis=1
        )

    else:

        df[f"Hit_{i}"] = 0


# Top-3 Hit
df["Top3_Hit"] = df[
    ["Hit_1", "Hit_2", "Hit_3"]
].max(axis=1)


# Top-3 Retrieval Accuracy
top3_accuracy = round(df["Top3_Hit"].mean(), 4)

print("Top-3 Retrieval Accuracy:", top3_accuracy)


# Save
df.to_excel(
    "minilm_rag_top3_accuracy_results.xlsx",
    index=False
)

Top-3 Retrieval Accuracy: 0.6
